## Coordinate coverage check

Checks how many trade corridors can be drawn on the flow map.

1. **Setup:** adds the project root to the import path so `src.country_coords` can be imported. This works whether the notebook runs from `notebooks/` or from the root.
2. **Load and map:** reads the HS 87 corridor totals and uses `add_coords` to attach origin and destination coordinates (`o_lat`, `o_lon`, `d_lat`, `d_lon`) to each corridor.
3. **Coverage stats:**
   - How many corridors have coordinates at both ends (these are the ones the map can draw).
   - What share of the total estimated trade value those corridors cover.
   - Which of the 20 largest corridors by value are missing coordinates.

**Result:** only 4,947 of the 12,661 corridors have full coordinates, but they cover **96.3%** of the trade value, and none of the top 20 are missing. The countries left out are small, so the map shows almost all of the trade value.

In [1]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.country_coords import add_coords

totals = pd.read_parquet(ROOT / "data" / "processed" / "corridor_totals_hs87.parquet")
t = add_coords(totals)
have = t.dropna(subset=["o_lat", "o_lon", "d_lat", "d_lon"])

print(f"corridors with full coords: {len(have):,} / {len(t):,}")
print(f"value covered: {have['est_value'].sum() / t['est_value'].sum() * 100:.1f}%")
top20 = t.sort_values("est_value", ascending=False).head(20)
print("missing among top 20:", top20[top20[['o_lat','d_lat']].isna().any(axis=1)][['origin','dest']].values.tolist())

corridors with full coords: 4,947 / 12,661
value covered: 96.3%
missing among top 20: []
